# Projet 1 : Découverte de Databricks

## Objectif
Construire un pipeline de données météo complet pour apprendre les bases de Databricks, de l'ingestion à l'analyse.

## Source des données
API gratuite [Open-Meteo](https://open-meteo.com) : température et précipitations horaires sur les 7 derniers jours, pour 5 villes (Port-Louis, Paris, Lyon, Marseille, Lille).

## Architecture (bronze, silver, gold)
| Couche | Table | Rôle |
|---|---|---|
| Bronze | `meteo.bronze_weather` | Données brutes issues de l'API |
| Silver | `meteo.silver_weather` | Données nettoyées et typées |
| Gold | `meteo.gold_daily_weather` | Agrégats par ville et par jour |

## Notebooks
1. `01_bronze_meteo` : ingestion depuis l'API (ce notebook)
2. `02_silver_meteo` : nettoyage
3. `03_gold_meteo` : agrégats et analyses

In [0]:
import requests

r = requests.get("https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": 48.85,
        "longitude": 2.35,
        "hourly": "temperature_2m,precipitation",
        "past_days": 1,
    },
    timeout=30,
)
print(r.status_code)
print(r.json()["hourly"]["temperature_2m"][:5])

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.meteo")
spark.sql("USE workspace.meteo")
print(spark.sql("SELECT current_catalog(), current_schema()").collect())

In [0]:
import requests

VILLES = {
    "Port-Louis": (-20.16, 57.50),
    "Paris": (48.85, 2.35),
    "Lyon": (45.76, 4.84),
    "Marseille": (43.30, 5.37),
    "Lille": (50.63, 3.06),
}

rows = []
for nom, (lat, lon) in VILLES.items():
    r = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "hourly": "temperature_2m,precipitation",
            "past_days": 7,
        },
        timeout=30,
    )
    r.raise_for_status()
    h = r.json()["hourly"]
    for t, temp, pr in zip(h["time"], h["temperature_2m"], h["precipitation"]):
        rows.append((nom, t, temp, pr))

df = spark.createDataFrame(rows, ["ville", "heure", "temperature", "precipitation"])
df.write.mode("overwrite").saveAsTable("meteo.bronze_weather")
print(f"{df.count()} lignes écrites dans meteo.bronze_weather")

In [0]:
display(spark.sql("SELECT ville, COUNT(*) AS nb FROM meteo.bronze_weather GROUP BY ville"))